###Preparation for SCD

In [0]:
dbutils.widgets.text("catalog", "dbr_dev_ua5816bd")
dbutils.widgets.text("silver_schema", "viktoriia_kalenichenko_silver")
dbutils.widgets.text("scd1_table", "dim_country_scd1")
dbutils.widgets.text("scd2_table", "dim_country_scd2_v2")

In [0]:
catalog = dbutils.widgets.get("catalog")
silver_schema = dbutils.widgets.get("silver_schema")

scd1_table = f"{catalog}.{silver_schema}.{dbutils.widgets.get('scd1_table')}"
scd2_table = f"{catalog}.{silver_schema}.{dbutils.widgets.get('scd2_table')}"

In [0]:
from pyspark.sql import functions as F
from delta.tables import DeltaTable

country_df = (
    spark.table(f"{catalog}.{silver_schema}.environment")
    .select("AREA","country_code")
    .dropDuplicates(["AREA"]))

In [0]:
#reference mapping (list)
country_regions = [
    ("Albania", "Europe", "Southern Europe"),
    ("Austria", "Europe", "Western Europe"),
    ("Belarus", "Europe", "Eastern Europe"),
    ("Belgium", "Europe", "Western Europe"),
    ("Bosnia and Herzegovina", "Europe", "Southern Europe"),
    ("Bulgaria", "Europe", "Eastern Europe"),
    ("Croatia", "Europe", "Southern Europe"),
    ("Czechia", "Europe", "Central Europe"),
    ("Denmark", "Europe", "Northern Europe"),
    ("Estonia", "Europe", "Northern Europe"),
    ("Finland", "Europe", "Northern Europe"),
    ("France", "Europe", "Western Europe"),
    ("Germany", "Europe", "Western Europe"),
    ("Greece", "Europe", "Southern Europe"),
    ("Hungary", "Europe", "Central Europe"),
    ("Iceland", "Europe", "Northern Europe"),
    ("Ireland", "Europe", "Northern Europe"),
    ("Italy", "Europe", "Southern Europe"),
    ("Latvia", "Europe", "Northern Europe"),
    ("Lithuania", "Europe", "Northern Europe"),
    ("Luxembourg", "Europe", "Western Europe"),
    ("Malta", "Europe", "Southern Europe"),
    ("Moldova", "Europe", "Eastern Europe"),
    ("Montenegro", "Europe", "Southern Europe"),
    ("Netherlands", "Europe", "Western Europe"),
    ("North Macedonia", "Europe", "Southern Europe"),
    ("Norway", "Europe", "Northern Europe"),
    ("Poland", "Europe", "Central Europe"),
    ("Portugal", "Europe", "Southern Europe"),
    ("Romania", "Europe", "Eastern Europe"),
    ("Serbia", "Europe", "Southern Europe"),
    ("Slovak Republic", "Europe", "Central Europe"),
    ("Slovenia", "Europe", "Central Europe"),
    ("Spain", "Europe", "Southern Europe"),
    ("Sweden", "Europe", "Northern Europe"),
    ("Switzerland", "Europe", "Western Europe"),
    ("Ukraine", "Europe", "Eastern Europe"),
    ("United Kingdom", "Europe", "Northern Europe"),
]

In [0]:
#creating a df from reference mapping
regions_df = spark.createDataFrame(country_regions, ["AREA", "region", "subregion"])

In [0]:
#the basis for SCD1 and SCD2
country_df = (country_df.join(regions_df, on="AREA", how="left"))

In [0]:
assert country_df.filter(
    F.col("country_code").isNull()
).count() == 0, "NULL country_code"

assert (
    country_df.groupBy("country_code")
    .count()
    .filter("count > 1")
    .count()
) == 0, "Duplicate country_code in source"

###SCD Type 1 implementation

In [0]:
#change classification of some countries to show how scd works
scd1_source = (
    country_df
    .withColumn(
        "subregion",
        F.when(F.col("AREA") == "Poland", "Eastern Europe")
         .when(F.col("AREA") == "Austria", "Central Europe")
         .otherwise(F.col("subregion"))
    )
    .withColumn("updated_at", F.current_timestamp())
)

In [0]:
scd1_delta = DeltaTable.forName(spark, scd1_table)

(
    scd1_delta.alias("target")
    .merge(
        scd1_source.alias("source"),
        "target.country_code = source.country_code"
    )
    .whenMatchedUpdate(
        set={
            "AREA": "source.AREA",
            "region": "source.region",
            "subregion": "source.subregion",
            "updated_at": "current_timestamp()"
        }
    )
    .whenNotMatchedInsertAll()
    .execute()
)

### SCD Type 2 implementation


SCD Type 2 preserves history by closing the previous current record and inserting a new current version when an attribute changes.

In [0]:
##change classification of some countries to show how scd works
scd2_source = (
    country_df
    .withColumn(
        "subregion",
        F.when(F.col("AREA") == "Poland", "Eastern Europe")
         .when(F.col("AREA") == "Austria", "Central Europe")
         .otherwise(F.col("subregion"))
    )
)

In [0]:
target = DeltaTable.forName(spark, scd2_table)
current_df = target.toDF().filter("is_current = true")

change_time = spark.sql(
    "SELECT current_timestamp() AS ts"
).first()["ts"]

scd2_source = scd2_source.withColumn(
    "effective_at", F.lit(change_time)
)
difference = """
    NOT (source.AREA <=> target.AREA)
    OR NOT (source.region <=> target.region)
    OR NOT (source.subregion <=> target.subregion)
"""

changed = (
    scd2_source.alias("source")
    .join(
        current_df.alias("target"),
        on="country_code"
    )
    .filter(F.expr(difference))
    .select("source.*")
)

staged = (
    changed.selectExpr(
        "CAST(NULL AS STRING) AS merge_key", "*"
    )
    .unionByName(
        scd2_source.selectExpr(
            "country_code AS merge_key", "*"
        )
    )
)

(
    target.alias("target")
    .merge(
        staged.alias("source"),
        """
        target.country_code = source.merge_key
        AND target.is_current = true
        """
    )
    .whenMatchedUpdate(
        condition=difference,
        set={
            "is_current": "false",
            "valid_to": "source.effective_at"
        }
    )
    .whenNotMatchedInsert(
        values={
            "AREA": "source.AREA",
            "country_code": "source.country_code",
            "region": "source.region",
            "subregion": "source.subregion",
            "valid_from": "source.effective_at",
            "valid_to": "CAST(NULL AS TIMESTAMP)",
            "is_current": "true"
        }
    )
    .execute()
)